In [ ]:
!pip install -q torch torchaudio librosa soundfile datasets torchcodec
!pip install -q huggingface_hub
!pip install -q evaluate jiwer
!pip install -q sacrebleu

("IMPORTANT INSTALLATION DONE")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 42.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 4.6 MB/s eta 0:00:00


'IMPORTANT INSTALLATION DONE'

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchaudio
import librosa
import sentencepiece as spm
from datasets import load_dataset, Audio
import torchcodec
from jiwer import wer, cer
import sacrebleu
from tqdm import tqdm
import numpy as np
import math
from IPython.display import Audio as IPyAudio, display


print("IMPORTING DONE")

IMPORTING DONE


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ValueError: mount failed

In [ ]:
dataset = load_dataset("Purvaxxx/hindi_test_dataset")
print(dataset)

In [ ]:
def preprocess_audio(audio_dict, target_sr=16000, n_fft=1024, hop_length=256, n_mels=80, normalize="zscore"):
    audio = audio_dict["array"]
    sr = audio_dict["sampling_rate"]

    # Resample if needed
    if sr != target_sr:
        audio = librosa.resample(audio, orig_sr=sr, target_sr=target_sr)
        sr = target_sr

    # Normalize waveform (peak normalization)
    if np.max(np.abs(audio)) > 0:
        audio = audio / np.max(np.abs(audio))

    # Ensure minimum length
    if len(audio) < n_fft:
        audio = np.pad(audio, (0, n_fft - len(audio)), mode="constant")

    # Mel spectrogram
    melspec = librosa.feature.melspectrogram(
        y=audio, sr=sr, n_mels=n_mels, n_fft=n_fft, hop_length=hop_length
    )

    # Convert to log scale
    log_melspec = librosa.power_to_db(melspec, ref=np.max).astype(np.float32)

    # Normalization
    if normalize == "zscore":
        mean = np.mean(log_melspec)
        std = np.std(log_melspec) + 1e-9
        log_melspec = (log_melspec - mean) / std
    elif normalize == "minmax":
        min_val = np.min(log_melspec)
        max_val = np.max(log_melspec)
        log_melspec = (log_melspec - min_val) / (max_val - min_val + 1e-9)

    # Return (time, n_mels)
    return log_melspec.T

In [ ]:
def load_and_preprocess(audio_dict):
    audio_array = audio_dict["array"]
    sr = audio_dict["sampling_rate"]
    display(IPyAudio(audio_array, rate=sr))
    log_melspec = preprocess_audio(audio_dict)
    return log_melspec

In [ ]:
hindi_chars = [
    ' ', 'अ','आ','इ','ई','उ','ऊ','ऋ','ए','ऐ','ओ','औ','अं','अः',
    'क','ख','ग','घ','ङ','च','छ','ज','झ','ञ','ट','ठ','ड','ढ','ण',
    'त','थ','द','ध','न','प','फ','ब','भ','म','य','र','ल','व','श','ष','स','ह',
    'ा','ि','ी','ु','ू','े','ै','ो','ौ','ं','ः','्','।', ',', '.'
]

vocab = ["[PAD]", "[SOS]", "[EOS]"] + hindi_chars

# Build mapping tables
char_to_id = {ch: i for i, ch in enumerate(vocab)}
id_to_char = {i: ch for i, ch in enumerate(vocab)}

def encode(text):
    # Ignore characters not in vocab
    return [char_to_id["[SOS]"]] + [char_to_id[ch] for ch in text if ch in char_to_id] + [char_to_id["[EOS]"]]

def decode(ids):
    return "".join(
        [id_to_char[i] for i in ids if i not in (char_to_id["[SOS]"], char_to_id["[EOS]"], char_to_id["[PAD]"])]
    )

SOS_ID = char_to_id["[SOS]"]
EOS_ID = char_to_id["[EOS]"]

print(vocab)
print(len(vocab))


In [ ]:
# POSITIONAL ENCODING for ASR and NMT

class PositionalEncodingNMT(nn.Module):
    def __init__(self, d_model, max_len=1000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(torch.log(torch.tensor(10000.0)) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(1)
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x: (seq_len, batch, d_model)
        return x + self.pe[:x.size(0)]

class PositionalEncodingASR(nn.Module):
    def __init__(self, d_model, max_len=1000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-torch.log(torch.tensor(10000.0)) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # [1, max_len, d_model]
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


print("Positional Encoding Done for Both")

In [ ]:
# TRANSFORMER ASR MODEL
class TransformerASR(nn.Module):
    def __init__(self,
                 input_dim,
                 d_model,
                 nhead,
                 num_encoder_layers,
                 num_decoder_layers,
                 dim_feedforward,
                 dropout,
                 vocab_size,
                 max_encoder_len,
                 max_decoder_len,
                ):
        super().__init__()
        self.input_linear = nn.Linear(input_dim, d_model)
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncodingASR(d_model, max_len=max_encoder_len)
        self.pos_decoder = PositionalEncodingASR(d_model, max_len=max_decoder_len)

        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )

        self.fc_out = nn.Linear(d_model, vocab_size)

    def forward(self,
                src,
                tgt,
                src_key_padding_mask=None,
                tgt_key_padding_mask=None,
                memory_key_padding_mask=None,
                tgt_mask=None
               ):
        src = self.input_linear(src)
        src = self.pos_encoder(src)

        tgt = self.embedding(tgt)
        tgt = self.pos_decoder(tgt)

        output = self.transformer(
            src, tgt,
            src_key_padding_mask=src_key_padding_mask,
            tgt_key_padding_mask=tgt_key_padding_mask,
            memory_key_padding_mask=memory_key_padding_mask,
            tgt_mask=tgt_mask
        )

        output = self.fc_out(output)
        return output

    def generate(self, src, start_token_id, end_token_id, max_len=100):
        self.eval()
        with torch.no_grad():
            src = self.input_linear(src)
            src = self.pos_encoder(src)
            memory = self.transformer.encoder(src)

            batch_size = src.size(0)
            ys = torch.full((batch_size, 1), start_token_id, dtype=torch.long, device=src.device)

            for _ in range(max_len - 1):
                tgt = self.embedding(ys)
                tgt = self.pos_decoder(tgt)

                tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt.size(1)).to(src.device)
                output = self.transformer.decoder(tgt, memory, tgt_mask=tgt_mask)
                output = self.fc_out(output[:, -1, :])
                next_token = torch.argmax(output, dim=1, keepdim=True)
                ys = torch.cat([ys, next_token], dim=1)

                if torch.all(next_token.squeeze() == end_token_id):
                    break

        return ys
PAD_IDX = 0
class NMTTransformer(nn.Module):
    def __init__(self,
                 src_vocab_size,
                 tgt_vocab_size,
                 d_model,
                 nhead,
                 num_layers,
                 dim_feedforward,
                 dropout
                ):
        super().__init__()
        self.src_embed = nn.Embedding(src_vocab_size, d_model)
        self.tgt_embed = nn.Embedding(tgt_vocab_size, d_model)
        self.pos_enc = PositionalEncodingNMT(d_model)
        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_layers,
            num_decoder_layers=num_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout
        )
        self.fc_out = nn.Linear(d_model, tgt_vocab_size)

    def forward(self, src, tgt):

        # Masks
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt.size(1)).to(src.device)
        src_key_padding_mask = (src == PAD_IDX)
        tgt_key_padding_mask = (tgt == PAD_IDX)

        # Embeddings and positional encoding
        # format(batch, seq_len, d_model)
        src_emb = self.pos_enc(self.src_embed(src))
        tgt_emb = self.pos_enc(self.tgt_embed(tgt))

        # Transformer - (seq_len, batch, d_model)
        src_emb = src_emb.transpose(0, 1)
        tgt_emb = tgt_emb.transpose(0, 1)

        output = self.transformer(
            src_emb,
            tgt_emb,
            tgt_mask=tgt_mask,
            src_key_padding_mask=src_key_padding_mask,
            tgt_key_padding_mask=tgt_key_padding_mask,
            memory_key_padding_mask=src_key_padding_mask
        )

        return self.fc_out(output.transpose(0, 1))

print("NMT ARCHITECTURE DEFINED")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

asr_model = TransformerASR(
    input_dim=80,
    d_model=256,
    nhead=4,
    num_encoder_layers=3,
    num_decoder_layers=3,
    dim_feedforward=512,
    dropout=0.1,
    vocab_size = len(vocab),
    max_encoder_len=17362,
    max_decoder_len=2898
).to(device)

nmt_model = NMTTransformer(
    src_vocab_size = 4000,
    tgt_vocab_size = 4000,
    d_model=512,
    nhead=4,
    num_layers=6,
    dim_feedforward=512,
    dropout=0.1
).to(device)


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

Transformer_ASR = "/content/drive/MyDrive/VOICE2ENGLISH/transformer_asr_finetuned.pt"
Transformer_NMT = "/content/drive/MyDrive/VOICE2ENGLISH/checkpoint_epoch_60 (1).pt"
spm_src = "/content/drive/MyDrive/VOICE2ENGLISH/spm_src.model"
spm_tgt = "/content/drive/MyDrive/VOICE2ENGLISH/spm_tgt.model"

# ASR MODEL LOAD
asr_checkpoint = torch.load(Transformer_ASR, map_location=device)
asr_model.load_state_dict(asr_checkpoint["model_state_dict"])
asr_model.to(device)
asr_model.eval()
print("ASR model loaded.")

# NMT MODEL LOAD
nmt_checkpoint = torch.load(Transformer_NMT, map_location=device)
nmt_model.load_state_dict(nmt_checkpoint["model_state_dict"])
nmt_model.to(device)
nmt_model.eval()
print("NMT model loaded")

sp_src = spm.SentencePieceProcessor(model_file=spm_src)
sp_tgt = spm.SentencePieceProcessor(model_file=spm_tgt)
print("SentencePiece models loaded.")


In [ ]:
# DECODING
def greedy_decode(model, src, max_len, device):
    model.eval()
    src = src.to(device)

    src_key_padding_mask = (src == PAD_IDX)

    src_emb = model.pos_enc(model.src_embed(src)).transpose(0, 1)
    memory = model.transformer.encoder(src_emb, src_key_padding_mask=src_key_padding_mask)

    ys = torch.tensor([[SOS_ID]], dtype=torch.long, device=device)

    for _ in range(max_len):
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(ys.size(1)).to(device)
        tgt_emb = model.pos_enc(model.tgt_embed(ys)).transpose(0, 1)

        out = model.transformer.decoder(
            tgt_emb,
            memory,
            tgt_mask=tgt_mask,
            memory_key_padding_mask=src_key_padding_mask)

        out = model.fc_out(out.transpose(0, 1))
        next_token = out[:, -1, :].argmax(dim=-1).item()
        ys = torch.cat([ys, torch.tensor([[next_token]], device=device)], dim=1)
        if next_token == EOS_ID:
            break
    return ys.squeeze(0).tolist()

print("DONE")

print("DONE")

In [ ]:
def predict_from_mel(mel_np, model, device, SOS_ID, EOS_ID, max_len=150):
    mel = torch.from_numpy(mel_np).float().unsqueeze(0).to(device)
    pred_ids = model.generate(mel, start_token_id=SOS_ID, end_token_id=EOS_ID, max_len=150)[0].tolist()
    pred_text = decode(pred_ids)
    return pred_text

In [ ]:
def translate_text(text, nmt_model, sp_src, sp_tgt, device, max_len=150):

    src_ids = sp_src.encode(text, out_type=int)
    src_tensor = torch.tensor([src_ids], device=device)
    pred_ids = greedy_decode(nmt_model, src_tensor, max_len=max_len, device=device)

    vocab_size = sp_tgt.get_piece_size()  # dynamic vocab size
    pred_ids_filtered = [
        i for i in pred_ids
        if i not in [PAD_IDX, SOS_ID, EOS_ID] and 0 <= i < vocab_size
    ]

    # Decode IDs back to text
    translated_text = sp_tgt.decode(pred_ids_filtered)
    return translated_text


In [ ]:
def speech_to_translation(mel_np, asr_model, nmt_model, sp_src, sp_tgt, device, SOS_ID, EOS_ID):
    asr_text = predict_from_mel(mel_np, asr_model, device, SOS_ID, EOS_ID)
    translated_text = translate_text(asr_text, nmt_model, sp_src, sp_tgt, device)
    return asr_text, translated_text


In [ ]:
#TESTING
idx = random.randint(0, len(dataset["hindi"]) - 1)
sample = dataset["hindi"][idx]

# audio_path = sample["chunked_audio_filepath"]
mel = load_and_preprocess(sample["chunked_audio_filepath"])

mel_tensor = torch.from_numpy(mel).float().unsqueeze(0).to(device) # Removed this line

ASR_text, NMT_text = speech_to_translation(
    mel, asr_model, nmt_model, sp_src, sp_tgt, device, SOS_ID, EOS_ID
)

print(f"Source Text = : {ASR_text}")
print(f"English Text = : {NMT_text}")

print(f" Hindi : {sample['text']}")
print(f" English : {sample['en_text']}")

ground_asr = sample["text"]
asr_output = ASR_text

asr_wer = wer(ground_asr, asr_output)
print(f"ASR WER: {asr_wer:.4f}")



In [ ]:
# WER CALCULATION
ground_asr = sample["text"]
asr_output = ASR_text

asr_wer = wer(ground_asr, asr_output)
print(f"ASR WER: {asr_wer:.4f}")

In [ ]:
from IPython.display import Audio as IPyAudio, display
audio_sample1 = "/content/drive/MyDrive/VOICE2ENGLISH/Sample02.wav"
array, sr = librosa.load(audio_sample1, sr=16000)

display(IPyAudio(array, rate=sr))

In [ ]:
audio_dict = {
    "array": array,
    "sampling_rate": sr
}

mel = preprocess_audio(audio_dict)
mel_tensor = torch.tensor(mel).unsqueeze(0).float().to(device)
pred_ids = asr_model.generate(mel_tensor, start_token_id=SOS_ID, end_token_id=EOS_ID, max_len=150)[0].tolist()
ASR_text = decode(pred_ids)
print(f" Source Text": {ASR_text}")
nmt_text = translate_text(ASR_text, nmt_model, sp_src, sp_tgt, device)
print(f" English Text: {nmt_text}")